<a href="https://colab.research.google.com/github/sonalisalvi/careloop-no-show-guard/blob/main/04_genai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Clone the public IIT project repository

# Exact repository URL created by you
REPO_URL = (
    "https://github.com/sonalisalvi/careloop-no-show-guard.git"
)

# !git clone https://github.com/sonalisalvi/careloop-no-show-guard.git

import subprocess
from pathlib import Path

# Repository Location inside the colab runtime

PROJECT_DIR = Path("/content/careloop-no-show-guard")

# Clone the repository if it is not already present.

if PROJECT_DIR.exists():
   print("Repository folder already exists.")
   print("Updating it from GitHub...")

   subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True
    )
else:
  print("Cloning the repository...")

  subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )

print("\nRepository connection completed!")
print("Project location:", PROJECT_DIR)

Cloning the repository...

Repository connection completed!
Project location: /content/careloop-no-show-guard


In [4]:

from pathlib import Path

PROJECT_DIR = Path("/content/careloop-no-show-guard")

print("Repository exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nRepository contents:")
    for item in PROJECT_DIR.iterdir():
        print(item.name)

    data_folder = PROJECT_DIR / "data"
    print("\nData folder exists:", data_folder.exists())

    if data_folder.exists():
        print("\nFiles inside data folder:")
        for item in data_folder.iterdir():
            print(item.name)


Repository exists: True

Repository contents:
LICENSE
01_ingest_clean.ipynb
README.md
test_risk_scores.csv
02_queries.sql
load_data.sql
.git
rejected_rows.json
data
appointments_clean.csv
clinics_enriched.csv

Data folder exists: True

Files inside data folder:
clinics.csv
README.md
followup_notes.csv
appointments.csv


# **Step 1: Load the follow-up notes**

In [5]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/careloop-no-show-guard")

notes_path = PROJECT_DIR / "data" / "followup_notes.csv"

notes = pd.read_csv(notes_path)

print("Dataset loaded successfully!")
print("Shape:", notes.shape)

print("\nColumn names:")
print(notes.columns.tolist())

print("\nFirst 10 records:")
display(notes.head(10))

Dataset loaded successfully!
Shape: (30, 3)

Column names:
['note_id', 'appointment_id', 'note_text']

First 10 records:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


# **Create the Pydantic validation model**

In [6]:
from pydantic import BaseModel, Field, ValidationError

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

print("Pydantic model created successfully!")
print("\nPermitted root causes:")
for cause in ALLOWED_ROOT_CAUSES:
    print("-", cause)

Pydantic model created successfully!

Permitted root causes:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other


# **Install Gemini SDK**

In [7]:
%pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [8]:
import os
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")

if not api_key.strip():
    raise ValueError("API key cannot be empty.")

os.environ["GEMINI_API_KEY"] = api_key.strip()

print("Gemini API key stored securely in this Colab runtime.")

Enter your Gemini API key: ··········
Gemini API key stored securely in this Colab runtime.


In [20]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

# MODEL_NAME = "gemini-2.5-flash"
MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client initialised successfully!")
print("Model:", MODEL_NAME)

Gemini client initialised successfully!
Model: gemini-3.5-flash-lite


# **Create the system instruction and prompt**

In [21]:
SYSTEM_INSTRUCTION = """
You are a healthcare operations analyst working for CareLoop Clinics.

Your task is to analyse patient follow-up notes explaining missed
outpatient appointments.

Use only the following root-cause categories:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

Rules:
1. Select exactly one permitted root-cause category.
2. Do not invent facts or assume information not stated.
3. Understand Hindi and Hinglish notes where possible.
4. If the reason is unclear or does not fit another category,
   select other.
5. Confidence must be a number between 0 and 1.
6. Summary must be concise and no more than 25 words.
7. Recommend a practical, patient-friendly operational action.
8. Return only the requested structured JSON.
"""

def build_craft_prompt(note_text):

    return f"""
CONTEXT:
CareLoop Clinics operates outpatient clinics and wants to understand
why patients miss appointments.

ROLE:
Act as an experienced healthcare operations analyst.

ACTION:
Analyse the following patient follow-up note.
Identify the most likely root cause, estimate confidence,
summarise the explanation and recommend a practical action.

PATIENT NOTE:
{note_text}

FORMAT:
Return exactly these four fields:
- root_cause
- confidence
- summary
- recommended_action

Use only the permitted root-cause categories.

TONE:
Professional, neutral, concise and empathetic.
Do not blame the patient.
"""

print("CRAFT prompt created successfully!")
print("\nSystem instruction length:", len(SYSTEM_INSTRUCTION))
print("Prompt function ready:", callable(build_craft_prompt))

CRAFT prompt created successfully!

System instruction length: 785
Prompt function ready: True


# **Configure Gemini's response schema**

In [22]:
from google.genai import types

response_schema = {
    "type": "OBJECT",
    "properties": {
        "root_cause": {
            "type": "STRING"
        },
        "confidence": {
            "type": "NUMBER"
        },
        "summary": {
            "type": "STRING"
        },
        "recommended_action": {
            "type": "STRING"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ],
    "propertyOrdering": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

generation_config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    temperature=0,
    response_mime_type="application/json",
    response_schema=response_schema,
    max_output_tokens=1000
)

print("Gemini JSON response configuration created!")
print("Temperature:", 0)
print("Response format: application/json")
print("Required fields:", response_schema["required"])

Gemini JSON response configuration created!
Temperature: 0
Response format: application/json
Required fields: ['root_cause', 'confidence', 'summary', 'recommended_action']


# **Create the analyse_note() function**

In [23]:
import json

def analyse_note(note_text):

    # 1. Send the note to Gemini
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(note_text),
        config=generation_config
    )

    # 2. Preserve the original response
    raw_response = response.text

    # 3. Convert JSON text into a Python dictionary
    result = json.loads(raw_response)

    # 4. Validate the response structure and data types
    insight = FollowupInsight.model_validate(result)

    # 5. Validate the root-cause category
    if insight.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(
            f"Invalid root cause: {insight.root_cause}"
        )

    # 6. Return validated result and original JSON
    return insight, raw_response


print("analyse_note() function created successfully!")

analyse_note() function created successfully!


# **Test Gemini using N001**

N001 says the patient never received a reminder message and forgot the appointment.

Our expected classification is **forgot_or_no_reminder**.

In [17]:
MODEL_NAME = "gemini-3.8-flash"

print("Updated Gemini model:", MODEL_NAME)

Updated Gemini model: gemini-3.8-flash


In [25]:
# Select the first patient note
test_note = notes.loc[
    notes["note_id"] == "N001",
    "note_text"
].iloc[0]

# Analyse using Gemini
test_insight, test_raw_response = analyse_note(test_note)

print("Patient note:")
print(test_note)

print("\nGemini raw JSON response:")
print(test_raw_response)

print("\nValidated AI result:")
print(test_insight.model_dump())

Patient note:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Gemini raw JSON response:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient missed the appointment because she did not receive a reminder message and forgot the schedule.",
  "recommended_action": "Verify patient contact details, update communication preferences, and assist with rescheduling for next week."
}

Validated AI result:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she did not receive a reminder message and forgot the schedule.', 'recommended_action': 'Verify patient contact details, update communication preferences, and assist with rescheduling for next week.'}


In [19]:
available_models = []

for model in client.models.list():
    methods = getattr(model, "supported_actions", []) or []

    if "generateContent" in methods:
        available_models.append(model.name)

print("Models supporting generateContent:")
for model in available_models:
    print(model)

Models supporting generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026


# **Analyse the first 10 patient notes**

In [26]:
import time
import json
import pandas as pd

analysis_results = []

for _, row in notes.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"\nProcessing {note_id}...")

    # Handle blank notes without calling Gemini
    if pd.isna(note_text) or not str(note_text).strip():

        analysis_results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

        print("Skipped: blank note")
        continue

    # Reuse our already successful N001 response
    if note_id == "N001":

        insight = test_insight
        raw_response = test_raw_response

    else:
        raw_response = ""

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=build_craft_prompt(note_text),
                config=generation_config
            )

            raw_response = response.text

            result = json.loads(raw_response)

            insight = FollowupInsight.model_validate(result)

            if insight.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError("Invalid root-cause category")

            # Pause before the next API request
            time.sleep(13)

        except Exception as e:

            analysis_results.append({
                "note_id": note_id,
                "appointment_id": appointment_id,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "Analysis requires manual review",
                "recommended_action": "Review patient note manually",
                "validation_status": "error",
                "raw_response": raw_response
            })

            print("Error:", str(e)[:250])
            time.sleep(13)
            continue

    # Store the validated response
    analysis_results.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": insight.root_cause,
        "confidence": insight.confidence,
        "summary": insight.summary,
        "recommended_action": insight.recommended_action,
        "validation_status": "success",
        "raw_response": raw_response
    })

    print("Root cause:", insight.root_cause)
    print("Confidence:", insight.confidence)

print("\nFirst 10 notes processed.")
print("Total records:", len(analysis_results))


Processing N001...
Root cause: forgot_or_no_reminder
Confidence: 0.98

Processing N002...
Root cause: distance_or_transport
Confidence: 0.95

Processing N003...
Root cause: cost_or_fee
Confidence: 0.95

Processing N004...
Skipped: blank note

Processing N005...
Root cause: illness_or_emergency
Confidence: 0.95

Processing N006...
Root cause: work_or_schedule
Confidence: 0.95

Processing N007...
Root cause: forgot_or_no_reminder
Confidence: 0.95

Processing N008...
Root cause: distance_or_transport
Confidence: 0.95

Processing N009...
Root cause: other
Confidence: 0.5

Processing N010...
Root cause: work_or_schedule
Confidence: 0.95

First 10 notes processed.
Total records: 10


# **Create and validate notes_analysed.csv**

In [27]:
# Convert results into a DataFrame
analysis_df = pd.DataFrame(analysis_results)

# Required output columns
required_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

analysis_df = analysis_df[required_columns]

# Expected categories for manual agreement checking
expected_categories = {
    "N001": "forgot_or_no_reminder",
    "N002": "distance_or_transport",
    "N003": "cost_or_fee",
    "N005": "illness_or_emergency",
    "N006": "work_or_schedule",
    "N007": "forgot_or_no_reminder",
    "N008": "distance_or_transport",
    "N009": "other",
    "N010": "work_or_schedule"
}

successful = analysis_df[
    analysis_df["validation_status"] == "success"
].copy()

successful["expected_root_cause"] = (
    successful["note_id"].map(expected_categories)
)

successful["agreement"] = (
    successful["root_cause"]
    == successful["expected_root_cause"]
)

# Save CSV in repository root
output_path = PROJECT_DIR / "notes_analysed.csv"
analysis_df.to_csv(output_path, index=False)

# Validation
print("Output shape:", analysis_df.shape)
print("\nValidation status:")
print(analysis_df["validation_status"].value_counts())

print("\nAgreement:", successful["agreement"].sum(),
      "/", len(successful))

print("\nSaved file:", output_path)
print("\nFirst 10 analysed records:")
display(analysis_df[
    ["note_id", "root_cause", "confidence", "validation_status"]
])

Output shape: (10, 8)

Validation status:
validation_status
success          9
skipped_blank    1
Name: count, dtype: int64

Agreement: 9 / 9

Saved file: /content/careloop-no-show-guard/notes_analysed.csv

First 10 analysed records:


,note_id,root_cause,confidence,validation_status
0,N001,forgot_or_no_reminder,0.98,success
1,N002,distance_or_transport,0.95,success
2,N003,cost_or_fee,0.95,success
3,N004,other,0.00,skipped_blank
4,N005,illness_or_emergency,0.95,success
5,N006,work_or_schedule,0.95,success
6,N007,forgot_or_no_reminder,0.95,success
7,N008,distance_or_transport,0.95,success
8,N009,other,0.50,success
9,N010,work_or_schedule,0.95,success


# **Test invalid AI responses**

Before trusting AI-generated results, we should test what happens when the model returns incorrect data. This is called **validation testing**.

**Your Pydantic model should reject:**
1. A confidence score of 1.7 because confidence must be between 0 and 1.
2. A response missing recommended_action because that field is required.

In [28]:
from pydantic import ValidationError

# Test 1: Invalid confidence score
bad_response_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send a reminder before the next appointment."
}

# Test 2: Missing required field
bad_response_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.85,
    "summary": "Patient had difficulty travelling to the clinic."
}

test_cases = [
    ("Invalid confidence", bad_response_1),
    ("Missing required field", bad_response_2)
]

for test_name, bad_response in test_cases:
    try:
        FollowupInsight.model_validate(bad_response)
        print(f"{test_name}: UNEXPECTEDLY PASSED")
    except ValidationError as e:
        print(f"{test_name}: Correctly rejected")
        print(e)

Invalid confidence: Correctly rejected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Missing required field: Correctly rejected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...velling to the clinic.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


# **connect AI findings with appointment data**
Now we'll answer a more useful business question:

Are certain reasons for missing appointments associated with patients who did not receive reminders?

We'll join the AI-generated root causes to the cleaned appointment dataset using appointment_id

In [29]:
# Load the cleaned appointment dataset
appointments_clean = pd.read_csv(
    PROJECT_DIR / "appointments_clean.csv"
)

# Merge AI analysis with appointment information
notes_with_appointments = analysis_df.merge(
    appointments_clean,
    on="appointment_id",
    how="left"
)

print("Merged shape:", notes_with_appointments.shape)
print("\nMissing appointment matches:")
print(notes_with_appointments["status"].isna().sum())

# Compare root causes with reminder status
print("\nRoot cause vs reminder status:")
display(
    pd.crosstab(
        notes_with_appointments["root_cause"],
        notes_with_appointments["reminder_sent"],
        margins=True
    )
)

Merged shape: (10, 23)

Missing appointment matches:
0

Root cause vs reminder status:


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10


**We'll compare V1 (simple prompt) with V2 (your CRAFT prompt)** on the same two notes, using the same Gemini model.
- N007 tests understanding of Hindi/Hinglish.
- N009 tests an ambiguous situation where the reason is not fully clear.
We'll compare the outputs for category correctness, confidence, and usefulness of the recommended action. We won't change your existing analysis_results or notes_analysed.csv

In [31]:
# V1: Simple prompt configuration (no detailed system instruction)
v1_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=response_schema,
    max_output_tokens=1000
)

# Two test notes
comparison_notes = {
    "N007": notes.loc[notes["note_id"] == "N007", "note_text"].iloc[0],
    "N009": notes.loc[notes["note_id"] == "N009", "note_text"].iloc[0]
}

comparison_results = []

for note_id, note_text in comparison_notes.items():

    print(f"\nTesting {note_id}")

    # V1: Simple prompt
    v1_prompt = f"""
    Classify the following patient follow-up note into the most
    appropriate root cause category.

    Allowed categories:
    {ALLOWED_ROOT_CAUSES}

    Return the required JSON fields.

    Patient note: {note_text}
    """

    v1_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=v1_prompt,
        config=v1_config
    )

    v1_data = json.loads(v1_response.text)
    v1_insight = FollowupInsight.model_validate(v1_data)

    comparison_results.append({
        "note_id": note_id,
        "prompt_version": "V1_Simple",
        "root_cause": v1_insight.root_cause,
        "confidence": v1_insight.confidence,
        "summary": v1_insight.summary,
        "recommended_action": v1_insight.recommended_action
    })

    time.sleep(13)

    # V2: Your existing CRAFT prompt
    v2_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(note_text),
        config=generation_config
    )

    v2_data = json.loads(v2_response.text)
    v2_insight = FollowupInsight.model_validate(v2_data)

    if v2_insight.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError("Invalid root cause in V2 response")

    comparison_results.append({
        "note_id": note_id,
        "prompt_version": "V2_CRAFT",
        "root_cause": v2_insight.root_cause,
        "confidence": v2_insight.confidence,
        "summary": v2_insight.summary,
        "recommended_action": v2_insight.recommended_action
    })

    print("V1:", v1_insight.root_cause)
    print("V2:", v2_insight.root_cause)

    time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)


Testing N007
V1: forgot_or_no_reminder
V2: forgot_or_no_reminder

Testing N009
V1: other
V2: other


,note_id,prompt_version,root_cause,confidence,summary,recommended_action
0,N007,V1_Simple,forgot_or_no_reminder,0.95,The patient forgot the appointment because the...,Reschedule the appointment and verify the pati...
1,N007,V2_CRAFT,forgot_or_no_reminder,0.95,Patient forgot the appointment as no reminder ...,Verify patient mobile contact details and ensu...
2,N009,V1_Simple,other,0.80,The patient could not be reached after three a...,Try contacting the patient at a later time or ...
3,N009,V2_CRAFT,other,0.50,Patient could not be reached as their phone wa...,Send an automated follow-up SMS and postal rem...


In [32]:
display(
    pd.DataFrame(comparison_results)
    [["note_id", "prompt_version", "root_cause", "confidence"]]
)

,note_id,prompt_version,root_cause,confidence
0,N007,V1_Simple,forgot_or_no_reminder,0.95
1,N007,V2_CRAFT,forgot_or_no_reminder,0.95
2,N009,V1_Simple,other,0.80
3,N009,V2_CRAFT,other,0.50


In [33]:
import time

time.sleep(30)

n009_text = comparison_notes["N009"]

try:
    v2_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_craft_prompt(n009_text),
        config=generation_config
    )

    v2_data = json.loads(v2_response.text)
    v2_insight = FollowupInsight.model_validate(v2_data)

    if v2_insight.root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError("Invalid root cause")

    comparison_results.append({
        "note_id": "N009",
        "prompt_version": "V2_CRAFT",
        "root_cause": v2_insight.root_cause,
        "confidence": v2_insight.confidence,
        "summary": v2_insight.summary,
        "recommended_action": v2_insight.recommended_action
    })

    print("N009 V2 completed successfully.")

except Exception as e:
    print("N009 V2 unavailable:", str(e)[:300])

display(pd.DataFrame(comparison_results))

N009 V2 completed successfully.


,note_id,prompt_version,root_cause,confidence,summary,recommended_action
0,N007,V1_Simple,forgot_or_no_reminder,0.95,The patient forgot the appointment because the...,Reschedule the appointment and verify the pati...
1,N007,V2_CRAFT,forgot_or_no_reminder,0.95,Patient forgot the appointment as no reminder ...,Verify patient mobile contact details and ensu...
2,N009,V1_Simple,other,0.80,The patient could not be reached after three a...,Try contacting the patient at a later time or ...
3,N009,V2_CRAFT,other,0.50,Patient could not be reached as their phone wa...,Send an automated follow-up SMS and postal rem...
4,N009,V2_CRAFT,other,0.50,Patient could not be reached as their phone wa...,Send an automated SMS notification inviting th...


In [34]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.drop_duplicates(
    subset=["note_id", "prompt_version"],
    keep="last"
).reset_index(drop=True)

comparison_results = comparison_df.to_dict("records")

print("Final number of rows:", len(comparison_df))
display(comparison_df)

Final number of rows: 4


,note_id,prompt_version,root_cause,confidence,summary,recommended_action
0,N007,V1_Simple,forgot_or_no_reminder,0.95,The patient forgot the appointment because the...,Reschedule the appointment and verify the pati...
1,N007,V2_CRAFT,forgot_or_no_reminder,0.95,Patient forgot the appointment as no reminder ...,Verify patient mobile contact details and ensu...
2,N009,V1_Simple,other,0.80,The patient could not be reached after three a...,Try contacting the patient at a later time or ...
3,N009,V2_CRAFT,other,0.50,Patient could not be reached as their phone wa...,Send an automated SMS notification inviting th...
